# Canonical Model 04: PEST Calibration and Results

This notebook calibrates the **canonical valley model itself**. The model's hydraulic conductivity is reset to a wrong starting value (3x too transmissive); head observations are sampled from the true K field, so there is real work for history matching to do.

We start the calibration with `model.pest(...)` and drive it through the declarative facade -- `parameterize`, `observe`, `forecast`, `build` -- which compiles straight to native `pyemu.utils.PstFrom`, run PESTPP-IES, and then review **how well the calibrated ensemble reproduces the observations**. (Notebook 06 takes the same setup further into forecast *uncertainty*.)

In [1]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header
from myflopy.modflow.mf6.canonical_calibration import build_canonical_calibration_demo

notebook_header('04', 'PEST Calibration and Results',
                'Calibrate the canonical model with PESTPP-IES, then review the fit.')

## 1. Build the calibration demo and declare the PEST problem

`build_canonical_calibration_demo` builds the canonical model as the synthetic truth, samples head observations across the valley floor (plus one down-valley head forecast near the lake), and resets K to the wrong start. We then declare two parameters, the head observations, and the forecast, and build the control file with `noptmax=0` (evaluate once and compute residuals).

In [2]:
import shutil
artifact_root = Path('../artifacts/canonical_pest')
shutil.rmtree(artifact_root, ignore_errors=True)  # clean rebuild: a stale pest/ template inside the model dir makes PstFrom recurse
artifact_root.mkdir(parents=True, exist_ok=True)

demo = build_canonical_calibration_demo(artifact_root / 'model')

# No workspace=: defaults to <model workspace>/pest/canonical_pest, so the run
# lives beside the model and shows up in demo.model.pest_runs (last cell).
cal = demo.model.pest('canonical_pest', start_datetime='2024-01-01')
# One constant K multiplier (scales every layer's K file together) and one
# constant recharge multiplier. Bounds are multiplier factors; `physical`
# clamps the final model value so calibration cannot reach nonphysical K.
cal.parameterize('k',        style='constant', bounds=(0.05, 2.0), physical=(0.01, 300.0))
cal.parameterize('recharge', style='constant', bounds=(0.3, 3.0),  physical=(0.0, 1e-2))
cal.observe(demo.head_targets)
cal.forecast(demo.forecast_targets)
pst = cal.build('canonical_pest.pst', noptmax=0)
print(cal.settings())

VoronoiGrid initializing.
Voronoi grid initialized.


getting connectivity properties (iac, ja, cl12, hwva, nja)


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:278: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(


writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_gwf...
  writing model viz_prt_master...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...


    writing package sto...
    writing package chd...
INFORMATION: maxbound in ('', 'chd', 'dimensions') changed to 200 based on size of stress_period_data
    writing package ghb...
INFORMATION: maxbound in ('', 'ghb', 'dimensions') changed to 200 based on size of stress_period_data
    writing package rch...
    writing package wel...
INFORMATION: maxbound in ('', 'wel', 'dimensions') changed to 2 based on size of stress_period_data
    writing package drn...
INFORMATION: maxbound in ('', 'drn', 'dimensions') changed to 55 based on size of stress_period_data
    writing package lak...
    writing package sfr...


...

    Solving:  Stress period:     6    Time step:     1


    Solving:  Stress period:     6    Time step:     2


 
 Run end date and time (yyyy/mm/dd hh:mm:ss): 2026/06/24 11:25:54
 Elapsed run time:  5.525 Seconds
 
 Normal termination of simulation.

Success is:  True


PEST calibration: canonical_pest  (model: viz_prt_master)
  template : ..\artifacts\canonical_pest\model\viz_prt_master\pest\canonical_pest
  start    : 2024-01-01
  parameters (2):
    - k          style=constant    bounds=(0.05, 2.0) physical=(0.01, 300.0) transform=log
    - recharge   style=constant    bounds=(0.3, 3.0) physical=(0.0, 0.01) transform=log
  observations (1):
    - hds        kind=headtarget   n=96
  forecasts (1):
    - fore1      n=6
  built control file:
    npar=2 (groups=2)  nobs=102 (nonzero weight=96)  forecasts=6  noptmax=0


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\pyemu\utils\pst_from.py:1281: PyemuWarning: add_py_function(): _write_head_target_csv already in forward run python functions, not overriding here, original will be maintained


In [3]:
pd.Series({
    'cells': int(demo.model.vor.ncpl),
    'observation_wells': demo.head_targets.locations_gdf.shape[0],
    'head_observations': len(demo.head_targets.to_long()),
    'adjustable_parameters': pst.npar_adj,
    'nonzero_observations': pst.nnz_obs,
    'start_K_factor': demo.start_k_factor,
}, name='calibration problem')

cells                    2500.0
observation_wells          16.0
head_observations          96.0
adjustable_parameters       2.0
nonzero_observations       96.0
start_K_factor              3.0
Name: calibration problem, dtype: float64

## 2. Validate the forward run

Every PEST iteration calls `forward_run.py`. Running it once directly is the key setup check: it must apply the parameter multipliers, run MODFLOW, and regenerate the simulated-observation files. **What to look for:** a zero return code and a regenerated `hds_simulated_heads.csv`.

In [4]:
import subprocess
template = cal.template_workspace
result = subprocess.run([sys.executable, 'forward_run.py'], cwd=template,
                        capture_output=True, text=True)
assert result.returncode == 0, result.stdout + '\n' + result.stderr
pd.Series({
    'forward_run_returncode': result.returncode,
    'apply_list_and_array_pars_present': 'apply_list_and_array_pars' in (template / 'forward_run.py').read_text(),
    'simulated_heads_regenerated': (template / 'hds_simulated_heads.csv').exists(),
}, name='forward-run validation')

forward_run_returncode                  0
apply_list_and_array_pars_present    True
simulated_heads_regenerated          True
Name: forward-run validation, dtype: object

## 3. Run the calibration (PESTPP-IES)

`run_ies` configures the ensemble options, launches PESTPP-IES with parallel agents, and returns an `IesResults`. The forward model is package-heavy (~7 s per run) and IES fires hundreds of runs, so this is a *go-get-coffee* cell -- use parallel `workers` and a modest ensemble for the demo. Set `RUN_IES = False` to skip it.

In [5]:
# Saved outputs below are from a completed run. Set RUN_IES = True to
# regenerate them (PESTPP-IES fires hundreds of solves -- a 'go-get-coffee'
# cell; use parallel workers and a modest ensemble).
RUN_IES = False
REALS = 30
ITERATIONS = 3
WORKERS = 12         # parallel PESTPP-IES agents

if RUN_IES:
    ies = cal.run_ies(reals=REALS, iterations=ITERATIONS, workers=WORKERS)
    print(ies.settings)
else:
    ies = None
    print('Skipped: set RUN_IES = True to run PESTPP-IES.')


2026-06-24 11:26:17,102 - MainProcess - INFO - Reserved port 4527 for process 38304


PESTPP-IES run: canonical_pest
  workspace   : ..\artifacts\canonical_pest\model\viz_prt_master\pest\canonical_pest_ies_master
  realizations: 30
  noptmax     : 3   iterations on disk: [0, 1, 2, 3]
  observations: 96 nonzero-weight   forecasts: 6
  noise ensemble: yes
  ies options : {'ies_num_reals': 30}


## 4. Did the misfit drop? (phi convergence)

The first thing to check: did history matching reduce the objective function (phi)? Each faint line is one realization; the bold line is the ensemble mean. **What to look for:** a clear drop from the prior to the posterior, *without* the ensemble collapsing to a single line (which signals over-fitting).

In [6]:
if ies is not None:
    display(ies.plot_phi())                       # plotly
    display(ies.plot_phi(backend='matplotlib'))   # seaborn/matplotlib

<Figure size 700x400 with 1 Axes>

## 5. Observed versus simulated

Grey is the prior ensemble, blue the posterior, red the measured heads. **What to look for:** the posterior (blue) spread should bracket the red markers -- close enough to fit, but not implausibly narrow.

In [7]:
if ies is not None:
    display(ies.plot_vs_obs())

## 6. Forecast summary and review bundle

`forecasts()` summarizes prior -> posterior uncertainty for the down-valley head prediction. `report(...)` bundles phi convergence, the ensemble-vs-observation comparison, and the forecast histograms into one self-contained HTML file. (Notebook 06 dissects the forecast uncertainty in depth.)

In [8]:
if ies is not None:
    display(ies.forecasts())
    report_path = ies.report(artifact_root / 'calibration_review.html')
    print('wrote', report_path)

,prior_mean,prior_std,prior_p05,prior_p50,prior_p95,posterior_mean,posterior_std,posterior_p05,posterior_p50,posterior_p95,truth,uncertainty_reduction
forecast,,,,,,,,,,,,
oname:fore1_otype:lst_usecol:fore_lakehead_per:0,92.812023,1.115487,92.185757,92.307215,95.086254,94.241574,0.053599,94.167611,94.233694,94.317919,94.257655,0.951950
oname:fore1_otype:lst_usecol:fore_lakehead_per:1,93.329798,1.214084,92.605888,92.821525,95.816982,94.945632,0.055681,94.869305,94.937488,95.024585,94.962035,0.954137
oname:fore1_otype:lst_usecol:fore_lakehead_per:2,93.495918,1.374867,92.640253,92.938338,96.338170,95.385503,0.061225,95.303165,95.376814,95.470903,95.402811,0.955469
oname:fore1_otype:lst_usecol:fore_lakehead_per:3,93.513018,1.485900,92.591998,92.910953,96.606669,95.559539,0.066621,95.471524,95.550319,95.651025,95.577649,0.955164
oname:fore1_otype:lst_usecol:fore_lakehead_per:4,93.541209,1.509326,92.601568,92.933161,96.681488,95.623999,0.067988,95.534897,95.614664,95.716701,95.642142,0.954955
oname:fore1_otype:lst_usecol:fore_lakehead_per:5,93.493718,1.502004,92.558914,92.887193,96.615072,95.577747,0.066508,95.490918,95.568655,95.668135,95.595344,0.955721


wrote ..\artifacts\canonical_pest\calibration_review.html


## Interpretation checklist

- Confirm phi actually dropped and the ensemble did not collapse to a single line.
- Check the posterior brackets the measured heads without being implausibly narrow.
- A lower objective function is *evidence*, not the whole deliverable -- carry forward the **base** realization, never the lowest-phi one.
- For the prediction you care about, value the posterior *spread*, not just a reduced phi -- see **Notebook 06** for the full uncertainty analysis.

## Your PEST runs live with the model

Because the calibration workspace defaults to `<model workspace>/pest/<name>`,
every PEST run done on this model is discoverable straight from the model via
`model.pest_runs` (or `run.pest_runs` for a loaded run) -- no need to remember
where the master directories were written. Reopen any of them for the full
`IesResults` review **without re-running**.

In [9]:
# Discover every PEST run done on this model, and reopen one for review.
runs = demo.model.pest_runs
for run in runs:
    print(run)

this_run = next((r for r in runs if r.name == cal.name and r.kinds), None)
if this_run is not None:
    review = this_run.review()          # -> IesResults (identical to a fresh run_ies)
    display(review.plot_phi())
else:
    print('No completed PEST runs yet -- set RUN_IES = True above and re-run.')

<PestRun 'canonical_pest' on viz_prt_master: ies>
